# TP 12 — RNN (réseau récurrent simple)
**Deep Learning · Séquences**


| Exemple | Diapo | Problème |
|---|---|---|
| Exemple 1 | 89 | Régression : à partir des ventes de lundi, mardi et mercredi, prédire les ventes de jeudi |
| Exemple 2 | 90 | Classification : les ventes sont-elles en hausse sur ces 3 jours ? |

**Le contexte.** Une petite pâtisserie de quartier note chaque soir ses ventes de la journée :

| Jour | Lundi | Mardi | Mercredi | Jeudi |
|---|---|---|---|---|
| Ventes (TND) | 100 | 200 | 300 | **? (réel : 400)** |

Le gérant veut un modèle qui lit les ventes **dans l’ordre des jours** et répond à deux questions :
1. **Combien** vendrai-je jeudi ? → un nombre : **régression** ;
2. Mes ventes sont-elles **en hausse** ? → oui / non : **classification**.

C’est une **série temporelle** : l’ordre des jours compte (100, 200, 300 n’a pas le même sens que 300, 200, 100). C’est exactement ce que sait lire un RNN. On travaille en **centaines de TND** (100 TND → 1) pour garder des calculs simples à la main : la séquence lue par le réseau est donc $1, 2, 3$ et la valeur à prédire $4$.

**Objectifs** : programmer **en Python pur** la lecture de la séquence par la cellule RNN, puis l’entraînement de la couche de sortie (3 epochs) ; refaire le même entraînement avec **Keras** (`SimpleRNN`) ; puis **comparer** les deux implémentations pour valider le code.

## Rappel de cours

Un RNN lit une séquence **pas à pas** et garde une **mémoire** $h_t$ qui résume ce qu’il a déjà lu :
$$h_t = \tanh(W_x\,x_t + W_h\,h_{t-1} + b)$$
Les **mêmes poids** $W_x, W_h, b$ servent à chaque pas de temps (poids partagés). La dernière mémoire $h_3$ alimente une **couche de sortie** :
- régression : $\hat y = v\cdot h + c$, perte $(y - \hat y)^2$ ;
- classification : $p = \sigma(v\cdot h + c)$, perte $-\ln p$ (quand $y = 1$).

**Simplification des diapos** (pour calculer à la main) : $W_x = 1$, $W_h = 0{,}5$, $b = 0$, **sans tanh**, donc
$$h_t = 0{,}5\,h_{t-1} + x_t$$
La cellule est **fixée** : on n’entraîne que la couche de sortie $(v, c)$.

In [ ]:
import math

jours = ["Lundi", "Mardi", "Mercredi"]
ventes_TND = [100, 200, 300]                  # ventes réelles de la pâtisserie
ventes_jeudi_TND = 400                        # ce qu'on aurait voulu prédire

sequence = [v / 100 for v in ventes_TND]      # en centaines de TND : [1.0, 2.0, 3.0]
y_reg = ventes_jeudi_TND / 100                # cible de la régression : 4.0
y_cls = 1                                     # cible de la classification : 1 = ventes en hausse

for j, v, x in zip(jours, ventes_TND, sequence):
    print(f"{j:<9}: {v} TND  →  x = {x}")

Lundi    : 100 TND  →  x = 1.0
Mardi    : 200 TND  →  x = 2.0
Mercredi : 300 TND  →  x = 3.0


---
# Partie A — Implémentation from scratch (Python pur)

### Question 1 — Lire les ventes de lundi, mardi, mercredi avec la cellule RNN
**Réponse.** Le RNN lit un jour à la fois. Il part d’une mémoire vide $h_0 = 0$ (avant lundi, il ne sait rien), puis applique la même formule chaque jour :
- lundi : $h_1 = 0{,}5\times 0 + 1 = 1$
- mardi : $h_2 = 0{,}5\times 1 + 2 = 2{,}5$
- mercredi : $h_3 = 0{,}5\times 2{,}5 + 3 = 4{,}25$

$h_3 = 4{,}25$ résume les trois jours : la vente de mercredi compte entièrement, celle de mardi à moitié, celle de lundi au quart ($3 + 0{,}5\times2 + 0{,}25\times1 = 4{,}25$). Les jours récents pèsent plus que les anciens.

In [ ]:
def lire_sequence(xs, Wx=1.0, Wh=0.5, b=0.0):
    # Cellule RNN simplifiée (sans tanh) : h(t) = Wx·x(t) + Wh·h(t−1) + b
    h = 0.0                       # mémoire vide au départ
    etats = []                    # mémoire après chaque pas
    for x in xs:                  # lecture pas à pas, dans l'ordre
        h = Wx * x + Wh * h + b   # nouvelle mémoire = entrée + une partie de l'ancienne
        etats.append(h)
    return etats

etats = lire_sequence(sequence)
for t, h in enumerate(etats, start=1):
    print(f"t = {t} ({jours[t - 1]:<8}) : x = {sequence[t - 1]} → h{t} = {h:.4f}")

h_final = etats[-1]               # la dernière mémoire résume toute la séquence
print("Mémoire finale transmise à la couche de sortie : h =", h_final)

t = 1 (Lundi   ) : x = 1.0 → h1 = 1.0000
t = 2 (Mardi   ) : x = 2.0 → h2 = 2.5000
t = 3 (Mercredi) : x = 3.0 → h3 = 4.2500
Mémoire finale transmise à la couche de sortie : h = 4.25


### Question 2 — Exemple 1 : prédire les ventes de jeudi (régression, diapo 84)
La couche de sortie est $\hat y = v\cdot h + c$ (en centaines de TND), la perte $(y - \hat y)^2$ avec $y = 4$ (400 TND). Départ $v = c = 0$ ; $\eta = 0{,}02$ ; 3 epochs.

**Réponse.** Erreur en sortie : $\delta = -2(y - \hat y)$. Gradients : $\nabla v = \delta\,h$ et $\nabla c = \delta$. La mémoire $h = 4{,}25$ ne change pas (la cellule est fixée) : seuls $v$ et $c$ bougent.

Epoch 1 à la main : $\hat y = 0$, $\delta = -2\times4 = -8$, donc $v = 0 - 0{,}02\times(-8)\times4{,}25 = 0{,}68$ et $c = 0 - 0{,}02\times(-8) = 0{,}16$.

In [ ]:
def entrainer_regression(h, y, eta, epochs=3):
    v, c = 0.0, 0.0
    lignes = []
    for e in range(epochs + 1):               # epoch 0 = état de départ
        y_chap = v * h + c                    # prédiction
        perte = (y - y_chap) ** 2             # MSE
        lignes.append((e, v, c, y_chap, perte))
        delta = -2 * (y - y_chap)             # dérivée de la perte par rapport à ŷ
        v = v - eta * delta * h               # mise à jour (après la mesure)
        c = c - eta * delta
    return lignes

reg = entrainer_regression(h_final, y=y_reg, eta=0.02)
print("Epoch |    v       c    |   ŷ      perte")
for e, v, c, y_chap, perte in reg:
    print(f"  {e}   | {v:6.3f}  {c:6.3f} | {y_chap:6.3f}  {perte:7.3f}")
print(f"Prévision pour jeudi : {reg[-1][3] * 100:.0f} TND (réel : {ventes_jeudi_TND} TND)")

Epoch |    v       c    |   ŷ      perte
  0   |  0.000   0.000 |  0.000   16.000
  1   |  0.680   0.160 |  3.050    0.902
  2   |  0.842   0.198 |  3.774    0.051
  3   |  0.880   0.207 |  3.946    0.003
Prévision pour jeudi : 395 TND (réel : 400 TND)


**Lecture.** La prévision pour jeudi passe de 0 à 3,95, soit **395 TND** (réel : 400 TND), et la perte de 16 à presque 0.

### Question 3 — Exemple 2 : les ventes sont-elles en hausse ? (classification, diapo 85)
De 100 à 200 puis 300 TND, les ventes montent : la réponse attendue est « oui » ($y = 1$). Sortie $p = \sigma(v\cdot h + c)$, perte $-\ln p$ ; $\delta = p - y$, $\nabla v = \delta\,h$, $\nabla c = \delta$ ; $\eta = 0{,}1$.

**Réponse.** Même mémoire $h$, mais une **tête de classification**. Epoch 1 à la main : $p = \sigma(0) = 0{,}5$, $\delta = -0{,}5$, donc $v = 0{,}1\times0{,}5\times4{,}25 = 0{,}21$ et $c = 0{,}05$.

In [ ]:
def sigmoide(z):
    return 1 / (1 + math.exp(-z))

def entrainer_classification(h, y, eta, epochs=3):
    v, c = 0.0, 0.0
    lignes = []
    for e in range(epochs + 1):
        p = sigmoide(v * h + c)                               # probabilité « ventes en hausse »
        perte = -math.log(p) if y == 1 else -math.log(1 - p)  # log-loss
        lignes.append((e, v, c, p, perte))
        delta = p - y                                         # erreur en sortie
        v = v - eta * delta * h
        c = c - eta * delta
    return lignes

cls = entrainer_classification(h_final, y=y_cls, eta=0.1)
print("Epoch |    v       c    |   p      perte")
for e, v, c, p, perte in cls:
    print(f"  {e}   | {v:6.3f}  {c:6.3f} | {p:6.3f}  {perte:7.3f}")

Epoch |    v       c    |   p      perte
  0   |  0.000   0.000 |  0.500    0.693
  1   |  0.213   0.050 |  0.722    0.326
  2   |  0.331   0.078 |  0.815    0.204
  3   |  0.409   0.096 |  0.862    0.148


**Lecture.** La probabilité que les ventes soient en hausse monte de 0,5 à 0,86 et la perte baisse de 0,69 à 0,15. La même mémoire $h$ sert aux deux tâches : seule la couche de sortie change.

---
# Partie B — Le même entraînement avec Keras

### Question 4 — Construire la cellule RNN dans Keras
**Réponse.** Keras possède la couche `SimpleRNN`. Avec `activation=None` (pas de tanh), elle calcule exactement $h_t = W_x x_t + W_h h_{t-1} + b$. On impose $W_x = 1$, $W_h = 0{,}5$, $b = 0$ et on la **gèle** (`trainable=False`) : seule la couche `Dense` de sortie est entraînée, comme dans la diapo.

Forme des données pour un RNN : `(nombre de séquences, nombre de pas, valeurs par pas)`. Ici une seule séquence (la semaine de la pâtisserie), 3 jours, une vente par jour, donc `(1, 3, 1)`.

In [ ]:
import numpy as np
import keras
from keras import layers

X = np.array(sequence).reshape(1, 3, 1)   # [[[1.], [2.], [3.]]] : 1 séquence, 3 jours, 1 vente par jour

def creer_cellule():
    cellule = layers.SimpleRNN(1, activation=None, trainable=False)   # 1 neurone de mémoire, sans tanh, gelé
    cellule.build((None, 3, 1))
    cellule.set_weights([np.array([[1.0]]),     # W_x (entrée)
                         np.array([[0.5]]),     # W_h (mémoire)
                         np.array([0.0])])      # b
    return cellule

cellule = creer_cellule()
h_keras = float(keras.ops.convert_to_numpy(cellule(X))[0, 0])
print("Mémoire h calculée par Keras :", round(h_keras, 4))

Mémoire h calculée par Keras : 4.25


### Question 5 — Entraîner les deux têtes avec Keras
**Réponse.** Le modèle complet est `Input(3, 1)` → `SimpleRNN` (gelé) → `Dense(1)`. La couche `Dense` part de $v = c = 0$ (`zeros`). On utilise `SGD` avec le même η et la même perte qu’en Python pur : `mse` pour la régression, `binary_crossentropy` pour la classification. Un *callback* enregistre $(v, c)$ au départ puis à la fin de chaque epoch.

In [ ]:
class HistoriqueSortie(keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.vc = [self.lire_vc()]                 # (v, c) au départ
    def on_epoch_end(self, epoch, logs=None):
        self.vc.append(self.lire_vc())             # (v, c) après chaque epoch
    def lire_vc(self):
        k, c = self.model.layers[-1].get_weights() # dernière couche = Dense de sortie
        return float(k[0, 0]), float(c[0])

def entrainer_keras(y, eta, classification):
    model = keras.Sequential([
        keras.Input(shape=(3, 1)),
        creer_cellule(),
        layers.Dense(1, activation="sigmoid" if classification else None,
                     kernel_initializer="zeros", bias_initializer="zeros")])
    model.compile(optimizer=keras.optimizers.SGD(learning_rate=eta),
                  loss="binary_crossentropy" if classification else "mse")
    hist = HistoriqueSortie()
    model.fit(X, np.array([[y]]), epochs=3, batch_size=1, verbose=0, callbacks=[hist])
    return model, hist.vc

m_reg, vc_reg = entrainer_keras(y_reg, eta=0.02, classification=False)
m_cls, vc_cls = entrainer_keras(float(y_cls), eta=0.1, classification=True)

print("Régression     : (v, c) par epoch =", [tuple(round(a, 3) for a in t) for t in vc_reg])
print(f"                 prévision pour jeudi = {float(m_reg.predict(X, verbose=0)[0, 0]) * 100:.0f} TND")
print("Classification : (v, c) par epoch =", [tuple(round(a, 3) for a in t) for t in vc_cls])
print("                 p(ventes en hausse) =", round(float(m_cls.predict(X, verbose=0)[0, 0]), 3))

Régression     : (v, c) par epoch = [(0.0, 0.0), (0.68, 0.16), (0.842, 0.198), (0.88, 0.207)]
                 prévision pour jeudi = 395 TND
Classification : (v, c) par epoch = [(0.0, 0.0), (0.213, 0.05), (0.331, 0.078), (0.409, 0.096)]
                 p(ventes en hausse) = 0.862


---
# Partie C — Comparaison et validation

### Question 6 — Python pur et Keras donnent-ils les mêmes résultats ?
**Réponse.** On compare la mémoire $h$, puis $v$ et $c$ à chaque epoch, pour les deux exemples.

In [ ]:
import pandas as pd

lignes = []
for nom, pur, ker in [("régression", reg, vc_reg), ("classification", cls, vc_cls)]:
    for (e, v, c, _, _), (vk, ck) in zip(pur, ker):
        lignes.append((nom, e, v, vk, c, ck))
tableau = pd.DataFrame(lignes, columns=["Exemple", "Epoch", "v Python pur", "v Keras", "c Python pur", "c Keras"])
print(f"Mémoire h : Python pur = {h_final:.4f} ; Keras = {h_keras:.4f}")
display(tableau.round(4))

assert abs(h_keras - h_final) < 1e-4
assert np.allclose([(v, c) for _, v, c, _, _ in reg], vc_reg, atol=1e-4)
assert np.allclose([(v, c) for _, v, c, _, _ in cls], vc_cls, atol=1e-4)
print("✅ Python pur = Keras : même mémoire h et mêmes paramètres de sortie à chaque epoch")

Mémoire h : Python pur = 4.2500 ; Keras = 4.2500


,Exemple,Epoch,v Python pur,v Keras,c Python pur,c Keras
0,régression,0,0.0000,0.0000,0.0000,0.0000
1,régression,1,0.6800,0.6800,0.1600,0.1600
2,régression,2,0.8415,0.8415,0.1980,0.1980
3,régression,3,0.8799,0.8799,0.2070,0.2070
4,classification,0,0.0000,0.0000,0.0000,0.0000
5,classification,1,0.2125,0.2125,0.0500,0.0500
6,classification,2,0.3308,0.3308,0.0778,0.0778
7,classification,3,0.4093,0.4093,0.0963,0.0963


✅ Python pur = Keras : même mémoire h et mêmes paramètres de sortie à chaque epoch


## Conclusion

| Ce qu’on a comparé (Python pur ↔ Keras) | Résultat |
|---|---|
| Mémoire $h$ après lecture de lundi, mardi, mercredi | ✅ identique (4,25) |
| Exemple 1 (ventes de jeudi, régression) : $v$ et $c$ à chaque epoch, prévision 395 TND | ✅ identiques |
| Exemple 2 (ventes en hausse ?, classification) : $v$ et $c$ à chaque epoch | ✅ identiques |

**À retenir**
- Un RNN lit la séquence pas à pas avec les **mêmes poids** à chaque pas ; $h_t$ est sa mémoire.
- Un RNN traite toute séquence : texte (chaque mot devient un vecteur), séries temporelles, signaux.
- La même mémoire peut alimenter une sortie de **régression** (linéaire + MSE) ou de **classification** (sigmoïde + log-loss).
- À chaque pas, l’ancienne mémoire est multipliée par $W_h$ : sur de longues séquences, le début s’efface et le gradient **s’évanouit**. C’est ce que corrigent le LSTM et le GRU.